# KrishiNexus · potato size sweep for `yolov11n-krishi-bd-v2.1` (run 2)

The app finds every potato in a heap photo, but a small potato gives the defect stage few pixels to work with. This
notebook measures how small a potato can be before the defect stage stops seeing what it sees on the same potato at
full size. The app warns on the certificate below the width chosen here.

- **Data:** all 477 photos of the Diamant ROI validation split. The defect test split stays closed.
- **Same cut as the app:** each labelled potato is cut out by its hand-drawn outline, the way `optical_grader._cascade`
  cuts a stage-1 outline. It is then green-boosted and segmented by run 2's `defect_best.pt` at 1024 px, conf 0.25.
  `green_boost` and `fill_outlines` below are the app's own functions, copied in verbatim.
- **Sweep:** each potato is graded at full size. Then the photo is shrunk (Lanczos, then saved as JPEG quality 90)
  until the potato is 48, 64, 96, 128, 192, 256, 384, 512 or 768 px wide, and graded again. Width means the diameter
  of a circle with the potato's area in pixels, as the app's `diameter_mm` uses.
- **Measure:** the share of potatoes whose per-potato grade (the BARI rule on blemish % and rot %) equals their
  full-size grade. Also reported: label agreement, grades turned milder or harsher, and blemish area kept.
- **Rule (set before the run):** the warning threshold is the smallest tested width at which, and at every larger
  tested width, at least 90% of potatoes keep their full-size grade. If no tested width qualifies, none is set.

**Data.** Tasnia, M. H. (2026). *Diamant Potato Dataset for ROI and Surface Defect Instance Segmentation*. Mendeley
Data, V1. doi:10.17632/4vty9rz7zw.1. Daffodil International University, CC BY 4.0.

**Before you run:** Settings → Internet → on (the data and `pip`). No accelerator is needed. Attach the output of
`krishinexus-potato-vision-training` (run 2) as an input.


## 1. Config


In [ ]:
import hashlib, io, json, math, os, pathlib, subprocess, sys, time, zipfile

WORK = "/kaggle/working"
RAW = "/tmp/diamant_raw"            # download and extracted zip (not kept)
DEFECT_IMGSZ, CONF, A_ROT = 1024, 0.25, 2.0   # the app's stage 2 and its rot limit (prompt registry "vision_grading")
BLEMISH_CLASSES, ROT_CLASSES = ['greening', 'mechanical_damage', 'scab'], ['rot']   # optical_grader's groups
SIZES = [48, 64, 96, 128, 192, 256, 384, 512, 768]   # potato width in px after shrinking
JPEG_QUALITY = 90                   # a shrunken photo is saved the way a phone saves one
KEEP = 0.90                         # set before the run: the share of potatoes that must keep their full-size grade
DEFECT_SHA256 = "84a079d7bbf331f42a6113c90e098fa9abe156b9693b4224acabf80aada6198d"   # run 2's defect_best.pt
ZIPS = {"roi": ("https://data.mendeley.com/public-files/datasets/4vty9rz7zw/files/187b5ef3-d642-4cbd-9cef-3dcab2d84ddd/file_downloaded",
                "27a32167f8c71b88c8ec88f80249b91a3a019ecc1992ed6e95f67fac262cea0d")}


## 2. Helpers (the vision notebook's, and the app's own cut)


In [ ]:
"""Helpers for the Diamant potato vision notebook: dataset layout, YOLO labels and the grading metrics."""
import pathlib

import numpy as np

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
SPLIT_NAMES = {"train": "train", "training": "train", "val": "val", "valid": "val", "validation": "val",
               "test": "test", "testing": "test"}
DEFECT_NAMES = ["scab", "rot", "greening", "mechanical_damage"]   # class ids 0-3, as published on the dataset page
BLEMISH, ROT = (0, 2, 3), (1,)   # the grader's two inputs: surface blemish feeds "bruising %", decay feeds "rot %"
GRADE_LIMITS = {"a_bruising": 3, "a_rot": 2, "b_bruising": 8, "b_rot": 5}   # prompt_registry "bari-standard"


def split_of(path):
    """The split named by the nearest directory in the path, or None."""
    for part in reversed(pathlib.PurePath(path).parts[:-1]):
        if part.lower() in SPLIT_NAMES:
            return SPLIT_NAMES[part.lower()]
    return None


def label_path(image):
    """YOLO convention: swap the nearest 'images' directory for 'labels' and use a .txt suffix."""
    parts = list(pathlib.PurePath(image).parts)
    for i in range(len(parts) - 2, -1, -1):
        if parts[i].lower() == "images":
            parts[i] = "labels"
            break
    return pathlib.Path(*parts).with_suffix(".txt")


def pair(files):
    """{split: [(image, label path or None)]} for every image, skipping macOS resource forks."""
    labels = {str(f) for f in files if f.suffix.lower() == ".txt"}
    out = {}
    for f in files:
        if f.suffix.lower() in IMAGE_EXTS and "__MACOSX" not in f.parts and not f.name.startswith("._"):
            lab = label_path(f)
            out.setdefault(split_of(f), []).append((f, lab if str(lab) in labels else None))
    return out


def read_label(text):
    """[(class_id, coords)] from a YOLO label: 4 numbers are a box, an even number above 4 a polygon."""
    rows = []
    for line in text.splitlines():
        tok = line.split()
        if not tok:
            continue
        nums = [float(t) for t in tok[1:]]
        if len(nums) < 4 or (len(nums) > 4 and len(nums) % 2):
            raise ValueError(f"bad YOLO label line: {line[:80]}")
        rows.append((int(float(tok[0])), np.array(nums, dtype=np.float32)))
    return rows


def polygon_mask(rows, classes, h, w):
    """Union mask of the labelled shapes whose class is in `classes`; boxes are drawn as rectangles."""
    from PIL import Image, ImageDraw

    img = Image.new("1", (w, h), 0)
    draw = ImageDraw.Draw(img)
    for cls, c in rows:
        if cls not in classes:
            continue
        if len(c) == 4:
            x, y, bw, bh = c
            c = np.array([x - bw / 2, y - bh / 2, x + bw / 2, y - bh / 2, x + bw / 2, y + bh / 2, x - bw / 2, y + bh / 2])
        pts = [(float(px) * w, float(py) * h) for px, py in c.reshape(-1, 2)]
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)
    return np.array(img, dtype=bool)


def union(masks, cls, classes, shape):
    """Union of the predicted masks whose class is in `classes` (an empty mask when there are none)."""
    keep = np.isin(cls, classes)
    return masks[keep].any(axis=0) if keep.any() else np.zeros(shape, dtype=bool)


def area_pct(mask, potato):
    """Share of the potato's visible pixels that the mask covers, in percent."""
    n = int(potato.sum())
    return 100.0 * float((mask & potato).sum()) / n if n else 0.0


def grade(bruising, rot, lim=GRADE_LIMITS):
    """The app's grade rule for one potato (uniformity is a lot-level measure, so it is left out)."""
    if bruising <= lim["a_bruising"] and rot <= lim["a_rot"]:
        return "A"
    if bruising <= lim["b_bruising"] and rot <= lim["b_rot"]:
        return "B"
    return "C"


def presence(gt_sets, pred_sets, n_classes):
    """Per class, precision/recall/F1 of the image-level call "this potato shows class c"."""
    out = {}
    for c in range(n_classes):
        tp = sum(c in g and c in p for g, p in zip(gt_sets, pred_sets))
        fp = sum(c not in g and c in p for g, p in zip(gt_sets, pred_sets))
        fn = sum(c in g and c not in p for g, p in zip(gt_sets, pred_sets))
        prec = tp / (tp + fp) if tp + fp else None
        rec = tp / (tp + fn) if tp + fn else None
        f1 = None if prec is None or rec is None else (2 * prec * rec / (prec + rec) if prec + rec else 0.0)
        out[c] = {"precision": prec, "recall": rec, "f1": f1, "support": tp + fn}
    return out


In [ ]:
import numpy as np


def green_boost(rgb: np.ndarray) -> np.ndarray:
    """The Diamant defect set's preprocessing: in OpenCV HSV, hue 35-85 with S and V >= 40 gets S x1.5 and V x1.3."""
    import cv2

    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    green = (hsv[..., 0] >= 35) & (hsv[..., 0] <= 85) & (hsv[..., 1] >= 40) & (hsv[..., 2] >= 40)
    hsv[..., 1] = np.where(green, np.minimum(hsv[..., 1] * 1.5, 255), hsv[..., 1])
    hsv[..., 2] = np.where(green, np.minimum(hsv[..., 2] * 1.3, 255), hsv[..., 2])
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)


def fill_outlines(shape: tuple[int, int], outlines: list[np.ndarray], offset: tuple[int, int] = (0, 0)) -> np.ndarray:
    """Rasterise pixel polygons (``Results.masks.xy``) into one mask; avoids full-size masks for 12 MP phone photos."""
    import cv2

    mask = np.zeros(shape, np.uint8)
    polygons = [np.round(o - offset).astype(np.int32) for o in outlines if len(o) >= 3]
    if polygons:
        cv2.fillPoly(mask, polygons, 1)
    return mask.astype(bool)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.208"], check=True)
import cv2, torch, ultralytics
from ultralytics import YOLO
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| opencv", cv2.__version__)


## 3. Data: download, verify, list the validation photos


In [ ]:
from collections import Counter
from PIL import Image, ImageOps

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 23), b""):
            h.update(block)
    return h.hexdigest()

pathlib.Path(RAW).mkdir(parents=True, exist_ok=True)
t0 = time.time()
for name, (url, digest) in ZIPS.items():
    zpath = pathlib.Path(RAW, f"{name}.zip")
    subprocess.run(["curl", "-L", "--fail", "--silent", "--show-error", "--retry", "5", "--retry-delay", "15",
                    "-o", str(zpath), url], check=True)
    got = sha256(zpath)
    assert got == digest, f"{name}.zip has sha256 {got}, but the dataset page publishes {digest}"
    with zipfile.ZipFile(zpath) as z:
        z.extractall(pathlib.Path(RAW, name))
    zpath.unlink()
    print(f"{name}: downloaded, sha256 verified, extracted ({time.time() - t0:.0f} s)")

files = lambda name: sorted(p for p in pathlib.Path(RAW, name).rglob("*") if p.is_file())
roi_val = sorted(pair(files("roi"))["val"], key=lambda p: p[0].name)
assert len(roi_val) == 477, len(roi_val)


## 4. The sweep


In [ ]:
weights = next(pathlib.Path("/kaggle/input").rglob("defect_best.pt"))
assert sha256(weights) == DEFECT_SHA256, f"{weights} is not run 2's defect_best.pt"
model = YOLO(str(weights))

def defects(rgb, outline, box=None):
    """optical_grader._cascade's stage 2 on one potato: (blemish %, rot %, potato pixels) from its outline."""
    h, w = rgb.shape[:2]
    if box is None:   # the app crops to stage 1's box; for a hand-drawn outline that is the outline's extent
        box = (*np.maximum(outline.min(0), 0), min(outline[:, 0].max(), w), min(outline[:, 1].max(), h))
    x1, y1, x2, y2 = (int(round(v)) for v in box)
    potato = fill_outlines((y2 - y1, x2 - x1), [outline], (x1, y1))
    if potato.sum() < 50:            # the app skips these too
        return None
    crop = green_boost(np.where(potato[..., None], rgb[y1:y2, x1:x2], 0).astype(np.uint8))
    res = model(Image.fromarray(crop), imgsz=DEFECT_IMGSZ, conf=CONF, verbose=False)[0]
    named = [] if res.masks is None else [(o, str(res.names[int(c)])) for o, c in zip(res.masks.xy, res.boxes.cls.tolist())]
    pct = [100.0 * float((fill_outlines(potato.shape, [o for o, n in named if n in group]) & potato).sum()) / float(potato.sum())
           for group in (BLEMISH_CLASSES, ROT_CLASSES)]
    return round(pct[0], 3), round(pct[1], 3), int(potato.sum())

def shrink(photo, f):
    """The photo shrunk by f and saved as a JPEG, as a phone saves a photo in which the potato is that small."""
    small = photo.resize((max(1, round(photo.width * f)), max(1, round(photo.height * f))), Image.LANCZOS)
    buf = io.BytesIO()
    small.save(buf, "JPEG", quality=JPEG_QUALITY)
    return Image.open(io.BytesIO(buf.getvalue())).convert("RGB")

potatoes, skipped = [], Counter()
t0 = time.time()
for i, (img, lab) in enumerate(roi_val):
    if lab is None:
        skipped["no label"] += 1
        continue
    photo = ImageOps.exif_transpose(Image.open(img)).convert("RGB")   # upright, as the app decodes photos
    rgb = np.asarray(photo)
    for k, (_, c) in enumerate(read_label(lab.read_text())):
        if len(c) <= 4:
            skipped["box, not outline"] += 1
            continue
        outline = c.reshape(-1, 2).astype(np.float64) * photo.size
        full = defects(rgb, outline)
        if full is None:
            skipped["under 50 px"] += 1
            continue
        width = 2 * math.sqrt(full[2] / math.pi)
        row = {"photo": img.name, "potato": k, "width_px": round(width), "full": list(full[:2]), "shrunk": {}}
        for s in SIZES:
            if s < width:
                small = shrink(photo, s / width)
                got = defects(np.asarray(small), outline * (small.width / photo.width, small.height / photo.height))
                if got is not None:
                    row["shrunk"][str(s)] = list(got[:2])
        potatoes.append(row)
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{len(roi_val)} photos, {len(potatoes)} potatoes, {time.time() - t0:.0f} s")
seconds = time.time() - t0
print(f"{len(potatoes)} potatoes in {seconds:.0f} s; skipped {dict(skipped)}")


## 5. Report


In [ ]:
import platform
import PIL

def label(blemish, rot):   # optical_grader._cascade's label rule without the size cut: shrinking keeps a potato's mm
    return "fungal_blight" if rot > A_ROT else "minor_bruise" if blemish > 0 else "uniform_A"

by_size = {}
for s in SIZES:
    pairs = [(p["full"], p["shrunk"][str(s)]) for p in potatoes if str(s) in p["shrunk"]]
    if not pairs:
        continue
    g = [(grade(*f), grade(*x)) for f, x in pairs]
    n = len(pairs)
    blemish = [x[0] / f[0] for f, x in pairs if f[0] >= 1.0]
    rotten = [x[1] > A_ROT for f, x in pairs if f[1] > A_ROT]
    by_size[s] = {
        "n": n,
        "grade_kept": round(sum(a == b for a, b in g) / n, 4),
        "label_kept": round(sum(label(*f) == label(*x) for f, x in pairs) / n, 4),
        "grade_milder": round(sum("ABC".index(b) < "ABC".index(a) for a, b in g) / n, 4),
        "grade_harsher": round(sum("ABC".index(b) > "ABC".index(a) for a, b in g) / n, 4),
        "blemish_area_kept_median": round(float(np.median(blemish)), 3) if blemish else None,   # potatoes >= 1% at full size
        "rot_kept": round(sum(rotten) / len(rotten), 4) if rotten else None,                     # potatoes > A_ROT at full size
        "full_to_shrunk_grade": {a: dict(Counter(b for aa, b in g if aa == a)) for a in "ABC"},
    }
    print(s, json.dumps({k: v for k, v in by_size[s].items() if k != "full_to_shrunk_grade"}))

qualifying = [s for s in by_size if all(by_size[t]["grade_kept"] >= KEEP for t in by_size if t >= s)]
threshold = min(qualifying) if qualifying else None
report = {
    "model": "yolov11n-krishi-bd-v2.1", "stage": "defect, run 2's chosen n1024", "weights_sha256": DEFECT_SHA256,
    "split": "val (ROI set, 477 photos); the defect test split is not opened",
    "cut": "hand-drawn potato outline, rasterised and green-boosted as optical_grader._cascade does",
    "shrink": f"Lanczos, then JPEG quality {JPEG_QUALITY}",
    "width": "diameter of a circle with the potato's pixel area",
    "rule": f"smallest tested width at which, and at every larger tested width, grade_kept >= {KEEP}",
    "min_potato_px": threshold,
    "n_potatoes": len(potatoes), "skipped": dict(skipped),
    "full_size_width_px": {q: int(np.percentile([p["width_px"] for p in potatoes], q)) for q in (5, 50, 95)},
    "full_size_grades": dict(Counter(grade(*p["full"]) for p in potatoes)),
    "by_size": by_size, "cpu_seconds": round(seconds),
    "dataset": {"name": "Diamant Potato Dataset", "doi": "10.17632/4vty9rz7zw.1", "licence": "CC BY 4.0",
                "sha256": {k: v[1] for k, v in ZIPS.items()}},
    "versions": {"python": platform.python_version(), "ultralytics": ultralytics.__version__, "torch": torch.__version__,
                 "numpy": np.__version__, "pillow": PIL.__version__, "opencv": cv2.__version__},
    "potatoes": potatoes,
}
json.dump(report, open(f"{WORK}/size_sweep.json", "w"), indent=1)
print("min_potato_px:", threshold)
